In [ ]:
from tests.adapters import run_get_response_log_probs, run_tokenize_prompt_and_output, run_sft_microbatch_train_step
from torch import Tensor
from tqdm import tqdm
import torch
import numpy as np
import random
import pandas as pd
from transformers import AutoModelForCausalLM, AutoTokenizer

In [ ]:
seed = 42
torch.manual_seed(seed)
np.random.seed(seed)
random.seed(seed)
device = torch.device("cuda")

num_epochs = 10
batch_size = 8
gradient_accumulation_steps = 4

In [ ]:
model = AutoModelForCausalLM.from_pretrained(
    'Qwen/Qwen2.5-Math-1.5B',
    torch_dtype=torch.bfloat16,
    attn_implementation='flash_attention_2'
).to(device)
tokenizer = AutoTokenizer.from_pretrained('Qwen/Qwen2.5-Math-1.5B')

In [ ]:
df = pd.read_parquet("short_reasoning_86.parquet")
prompts = df['prompt'].tolist()
responses = df['response'].tolist()
tokenized = run_tokenize_prompt_and_output(prompts, responses, tokenizer)
input_ids = tokenized['input_ids'].long().to(device)
labels = tokenized['labels'].long().to(device)
response_mask = tokenized['response_mask'].to(device)

In [ ]:
from torch.utils.data import DataLoader, Dataset

class MathSFTDataset(Dataset):
    def __init__(self, input_ids, labels, masks):
        self.input_ids = input_ids
        self.labels = labels
        self.masks = masks

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, index):
        input_id = self.input_ids[index]
        label = self.labels[index]
        mask = self.masks[index]
        return input_id, label, mask

train_loader = DataLoader(
    dataset=MathSFTDataset(input_ids, labels, response_mask),
    batch_size=batch_size
)


In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=5e-5, weight_decay=0.1)


In [ ]:
global_step = -1

for epoch in range(num_epochs):
    model.train()

    for idx, (input_batch, label_batch, mask) in tqdm(enumerate(train_loader)):
        log_probs = run_get_response_log_probs(model, input_batch, label_batch, False)['log_probs']
        loss, _ = run_sft_microbatch_train_step(log_probs, mask, gradient_accumulation_steps)
        global_step += 1
        
        if (idx + 1) % gradient_accumulation_steps == 0:
            optimizer.step()
            optimizer.zero_grad()
            
        print(f"Step {global_step:06d}: Train loss: {loss.cpu().item()}")


In [ ]:
import os

output_dir = "sft_model"
os.makedirs(output_dir, exist_ok=True)

print("saving the model and tokenizer...")
model.save_pretrained(save_directory=output_dir)
tokenizer.save_pretrained(save_directory=output_dir)